# NewsBot Intelligence System 2.0 — 07 System Integration

## Goal
Run the full capstone pipeline on new texts, batch input, corpus queries, and multilingual examples.

This notebook imports reusable project modules rather than duplicating implementation logic.

## Initialize once
Timing includes the local integrated components after model/index fitting.

In [1]:
from src.system import NewsBot2IntegratedSystem
system = NewsBot2IntegratedSystem().fit()
len(system.dataframe), system.search_engine.backend_status()

(1800,
 {'backend': 'tfidf',
  'model': None,
  'transformer_requested': False,
  'warning': None})

## Analyze new article texts
These short inputs are newly authored for the demo and are not training records.

In [2]:
new_articles = ['A software company announced an artificial intelligence product for small businesses in California.', 'City officials opened cooling centers after a dangerous heat wave and urged residents to check on neighbors.', 'A local sports club won a championship after a close game and celebrated with fans.']
analyses = [system.comprehensive_analysis(article) for article in new_articles]
[{'category': item['classification']['primary_category'], 'sentiment': item['sentiment']['label'], 'runtime': item['runtime'], 'seconds': item['statistics']['processing_seconds']} for item in analyses]

[{'category': 'BUSINESS',
  'sentiment': 'positive',
  'runtime': {'summarization': {'backend': 'extractive',
    'model': None,
    'transformer_requested': False,
    'warning': None},
   'retrieval': {'backend': 'tfidf',
    'model': None,
    'transformer_requested': False,
    'warning': None}},
  'seconds': 1.4381},
 {'category': 'BUSINESS',
  'sentiment': 'negative',
  'runtime': {'summarization': {'backend': 'extractive',
    'model': None,
    'transformer_requested': False,
    'warning': None},
   'retrieval': {'backend': 'tfidf',
    'model': None,
    'transformer_requested': False,
    'warning': None}},
  'seconds': 0.1859},
 {'category': 'SPORTS',
  'sentiment': 'positive',
  'runtime': {'summarization': {'backend': 'extractive',
    'model': None,
    'transformer_requested': False,
    'warning': None},
   'retrieval': {'backend': 'tfidf',
    'model': None,
    'transformer_requested': False,
    'warning': None}},
  'seconds': 0.1994}]

## Batch report and conversational query
Batch size is capped and responses identify historical scope.

In [3]:
report = system.generate_insights_report(new_articles, report_type='summary')
query = system.query_interface('Show me positive tech news from this week')
{'batch_report': report, 'query_response': query['response'], 'query_note': query['intent']['parameters'].get('timeframe_note')}

{'batch_report': {'report_type': 'summary',
  'articles_analyzed': 3,
  'category_distribution': {'BUSINESS': 2, 'SPORTS': 1},
  'sentiment_labels': {'positive': 2, 'negative': 1},
  'note': 'Corpus-derived descriptive insight; not independent fact verification.',
  'analyses': []},
 'query_response': "Found 0 matching historical articles with category=TECH, sentiment=positive, timeframe_note='This week' is resolved relative to the historical dataset maximum date (2022-09-14)..",
 'query_note': "'This week' is resolved relative to the historical dataset maximum date (2022-09-14)."}

## Multilingual integrated analysis
The translation field records availability and backend rather than hiding a fallback.

In [4]:
spanish = 'La empresa tecnológica anunció una nueva herramienta de inteligencia artificial para pequeñas empresas.'
multilingual_result = system.comprehensive_analysis(spanish)
{'language': multilingual_result['language'], 'translation': multilingual_result['translation'], 'warnings': multilingual_result['warnings']}

{'language': {'language': 'es',
  'language_name': 'Spanish',
  'confidence': 0.9999941005356101,
  'is_supported': True,
  'warning': None},
 'translation': {'translation': 'The technology company announced a new artificial intelligence tool for small businesses.',
  'available': True,
  'backend': 'authored_demo',
  'warning': None},
 'warnings': []}

## Final evidence and reflection
Use these stored metrics in the report and presentation, including their limitations.

In [5]:
import json
from pathlib import Path
metrics = json.loads((Path('data/results/metrics/evaluation_summary.json')).read_text())
{key: metrics[key] for key in ['classification','semantic_search','multilingual','conversation','runtime_backends']}

{'classification': {'selected_model': 'Multinomial Naive Bayes (midterm baseline)',
  'accuracy': 0.7138888888888889,
  'macro_precision': 0.7209936430170055,
  'macro_recall': 0.7138888888888889,
  'macro_f1': 0.7118910656946621,
  'weighted_f1': 0.7118910656946622,
  'calibration': {'expected_calibration_error': 0.3748121538866589,
   'mean_confidence': 0.33907673500223,
   'top_label_accuracy': 0.7138888888888889,
   'interpretation': 'Top-label expected calibration error on the held-out split; lower is better.'}},
 'semantic_search': {'precision_at_1': 0.6666666666666666,
  'precision_at_5': 0.7333333333333334,
  'hit_rate_at_5': 1.0,
  'queries': 6,
  'note': 'Category-based authored relevance evaluation; it measures topical retrieval, not factual agreement.'},
 'multilingual': {'examples': 6,
  'language_detection_accuracy': 1.0,
  'translation_availability': 1.0,
  'mean_translation_similarity': 1.0,
  'cross_lingual_retrieval_success': 1.0,
  'note': 'Spanish/French examples ar

## Limitations and ethics
The source corpus is historical and predominantly English. Confidence is not truth; sentiment, topics, named entities, translation, summaries, and semantic similarity can be wrong. Entity co-occurrence is not a proven real-world relationship. Internal corpus corroboration is not independent fact-checking.